In [1]:
#安裝匯入套件
# ! pip install seaborn
! pip install opencc
# ! pip install -U scikit-learn

import numpy as np
import pandas as pd
import torch
import torch.nn
import torch.nn.utils.rnn
import torch.utils.data
import matplotlib.pyplot as plt
import seaborn as sns
import opencc
import os
from sklearn.model_selection import train_test_split

data_path = '/work/claire901114/NLP_HW2' 

In [2]:
# 訓練與驗證集
df_train = pd.read_csv(os.path.join(data_path, 'arithmetic_train.csv'))
df_eval = pd.read_csv(os.path.join(data_path, 'arithmetic_eval.csv'))
df_train.head()

,src,tgt
0,14*(43+20)=,882
1,(6+1)*5=,35
2,13+32+29=,74
3,31*(3-11)=,-248
4,24*49+1=,1177


In [3]:
# 將數據轉換為字串格式
df_train['tgt'] = df_train['tgt'].apply(lambda x: str(x))
df_train['src'] = df_train['src'].add(df_train['tgt'])
df_train['len'] = df_train['src'].apply(lambda x: len(x))

df_eval['tgt'] = df_eval['tgt'].apply(lambda x: str(x))

In [4]:
# === 構建字元映射表 (Dictionary Building) ===
char_to_id = {}
id_to_char = {}

# 加特殊符號
char_to_id['<pad>'] = 0  # 用於填充序列
char_to_id['<eos>'] = 1  # 用於標示序列結束
id_to_char[0] = '<pad>'  # ID 0 對應 <pad>
id_to_char[1] = '<eos>'  # ID 1 對應 <eos>

# 收集資料集中的唯一字元
# 彙整訓練資料中出現過的所有字元，為建立詞彙表做準備。
unique_chars = set(''.join(df_train['src']))
# 移除已手動定義的特殊符號，避免重複分配 ID
unique_chars.discard('<pad>')
unique_chars.discard('<eos>')

# 穩定性優化：排序字元
# 透過排序確保每次執行程式時，字元與 ID 的對應關係保持一致，避免模型訓練的隨機性。
sorted_chars = sorted(list(unique_chars))

# 每個字符分配ID
current_id = 2  # 從 ID 2 開始，因為 0 和 1 被 <pad> 和 <eos> 使用
for char in sorted_chars:
    # 檢查該字元是否已經存在 
    if char not in char_to_id:
        char_to_id[char] = current_id  # 將字符對應到當前 ID
        id_to_char[current_id] = char  # 將 ID 對應到字符
        current_id += 1  

vocab_size = len(char_to_id)
print('Vocab size: {}'.format(vocab_size))

Vocab size: 18


In [5]:
# === 資料預處理：構建訓練序列與遮罩標籤 ===
# 將原始資料轉換為模型所需的輸入與輸出格式，並加入序列結束符號 <eos>。
def build_masked_target_ids(text, mapping, pad_id):
    """
    生成輸入 ID 序列與帶遮罩的目標 ID 序列。
    模型應專注於預測「等號後」的答案，而非重複輸入的算式。
    """
    # 1. 完整輸入 ID 序列 (含 <eos>)
    input_ids = [mapping.get(ch, pad_id) for ch in text] + [mapping['<eos>']]

    # 2. 標準的目標 ID 序列 (往左平移)
    # Target: I_1, I_2, ..., I_N, PAD
    target_ids = input_ids[1:] + [pad_id]

    # 3. 實作遮罩邏輯 (等號前的預測不計入loss(設為IGNORE_INDEX)，僅計算答案部分的 Loss)
    masked_target_ids = []

    # 找到等號的位置，並處理等號前的序列
    found_equal = False
    for i, ch in enumerate(text):
        if ch == '=':
            found_equal = True

        # 判斷當前 token 是否是答案的一部分（即等號後的第一個數字或 <eos>）
        # 目標序列的第 i 個元素對應的是輸入序列的第 i+1 個元素。

        if not found_equal:
            # 等號前，目標 ID 設為 IGNORE_INDEX (PAD_ID = 0)
            masked_target_ids.append(pad_id)
        else:
            # 等號之後，目標 ID 設為正常值
            masked_target_ids.append(target_ids[i])

    # 4. 處理 <eos> 的目標 ID
    # 序列的最後一個目標是 PAD_ID (target_ids 的最後一個元素)，它對應的是 <eos> 的輸入。
    # 確保 masked_target_ids 的長度與 target_ids 一致

    # 從 target_ids 的角度進行遮罩
    masked_target_ids = []

    # 標記等號在 input_ids中的位置
    equal_idx = -1
    try:
        equal_idx = input_ids.index(char_to_id['='])
    except ValueError:
        # 若算式中未包含等號（異常資料）
        pass

    for i in range(len(target_ids)):
        # 判定邏輯：i 對應的是模型預測 Input[i] 後產出的 Target[i]
        # 僅當目標位置處於等號之後（即預測結果為答案的一部分），才保留真實標籤
        if equal_idx != -1 and i >= equal_idx:
             # 如果目標是對應答案的 ID (在 '=' 之後)
            masked_target_ids.append(target_ids[i])
        else:
            # 等號前或等號本身的目標，設為 IGNORE_INDEX (PAD_ID = 0)
            masked_target_ids.append(pad_id)

    return input_ids, masked_target_ids


# 處理訓練集：生成特徵 (char_id_list)、標籤 (label_id_list) 並統計序列長度
results_train = df_train['src'].apply(lambda x: build_masked_target_ids(x, char_to_id, char_to_id['<pad>']))
df_train['char_id_list'] = [r[0] for r in results_train]
df_train['label_id_list'] = [r[1] for r in results_train]
df_train['len'] = df_train['char_id_list'].apply(len)

# 處理驗證集：確保驗證邏輯與訓練一致
results_eval = df_eval['src'].apply(lambda x: build_masked_target_ids(x, char_to_id, char_to_id['<pad>']))
df_eval['char_id_list'] = [r[0] for r in results_eval]
df_eval['label_id_list'] = [r[1] for r in results_eval]
df_eval['len'] = df_eval['char_id_list'].apply(len)

df_train.head()

,src,tgt,len,char_id_list,label_id_list
0,14*(43+20)=882,882,15,"[8, 11, 4, 2, 11, 10, 5, 9, 7, 3, 17, 15, 15, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 15, 15, 9, 1, 0]"
1,(6+1)*5=35,35,11,"[2, 13, 5, 8, 3, 4, 12, 17, 10, 12, 1]","[0, 0, 0, 0, 0, 0, 0, 10, 12, 1, 0]"
2,13+32+29=74,74,12,"[8, 10, 5, 10, 9, 5, 9, 16, 17, 14, 11, 1]","[0, 0, 0, 0, 0, 0, 0, 0, 14, 11, 1, 0]"
3,31*(3-11)=-248,-248,15,"[10, 8, 4, 2, 10, 6, 8, 8, 3, 17, 6, 9, 11, 15...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 9, 11, 15, 1, 0]"
4,24*49+1=1177,1177,13,"[9, 11, 4, 11, 16, 5, 8, 17, 8, 8, 14, 14, 1]","[0, 0, 0, 0, 0, 0, 0, 8, 8, 14, 14, 1, 0]"


In [6]:
batch_size = 64
epochs = 10
embed_dim = 256
hidden_dim = 256
lr = 0.001 
grad_clip = 1

In [7]:
# === 資料批次處理 (Data Batching) ===
# 運用 PyTorch 標準封裝，將預處理好的 ID 序列轉換為模型可高效讀取的 Batch 格式。

class Dataset(torch.utils.data.Dataset):
    """
    將 Pandas DataFrame 封裝為 PyTorch Dataset。
    """
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        # 回傳總資料筆數，讓 DataLoader 知道取樣範圍
        return len(self.sequences)

    def __getitem__(self, index):
        # 根據給定的索引 (index) 從資料集中提取一組訓練樣本
        # x: 輸入字元 ID 列表 (char_id_list)
        x = self.sequences.iloc[index]['char_id_list'] # Write your code here
        # y: 帶有損失遮罩的目標標籤 ID 列表 (label_id_list)
        y = self.sequences.iloc[index]['label_id_list'] # Write your code here
        return x, y

# collate function：用於動態處理 Batch 內的序列對齊
def collate_fn(batch):
    # 轉換為 Tensor 格式，為後續運算做準備
    batch_x = [torch.tensor(data[0]) for data in batch]
    batch_y = [torch.tensor(data[1]) for data in batch]

    # 記錄每個樣本的原始長度
    batch_x_lens = torch.LongTensor([len(x) for x in batch_x])
    batch_y_lens = torch.LongTensor([len(y) for y in batch_y])

    # Padding：每個 Batch中的句子長度不同， pad_sequence 補齊至該 Batch 的最大長度，才能以Tensor進行平行計算
    pad_batch_x = torch.nn.utils.rnn.pad_sequence(batch_x,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])

    pad_batch_y = torch.nn.utils.rnn.pad_sequence(batch_y,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])

    return pad_batch_x, pad_batch_y, batch_x

In [8]:
# 封裝訓練集資料
ds_train = Dataset(df_train[['char_id_list', 'label_id_list']])

In [9]:
# 建立數據加載器：負責打亂資料順序與 Batching
dl_train = torch.utils.data.DataLoader(
    dataset=ds_train,
    batch_size=batch_size,
    shuffle=True,       # 在訓練時打亂資料順序
    collate_fn=collate_fn
)

In [10]:
# === LSTM Model Design ===
class CharRNN(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CharRNN, self).__init__()

        # Embedding Layer
        # 將離散的字元 ID 映射為連續的向量空間，並設定 padding_idx 確保填充標記不參與梯度更新
        self.embedding = torch.nn.Embedding(num_embeddings=vocab_size,
                                            embedding_dim=embed_dim,
                                            padding_idx=char_to_id['<pad>'])

        # LSTM Layers：使用雙層 LSTM 結構以捕捉資料中更深層的序列依賴關係
        # batch_first=True 確保輸入張量維度為 [Batch, Seq, Feature]
        self.rnn_layer1 = torch.nn.LSTM(input_size=embed_dim,
                                        hidden_size=hidden_dim,
                                        batch_first=True)
        
        self.rnn_layer2 = torch.nn.LSTM(input_size=hidden_dim,
                                        hidden_size=hidden_dim,
                                        batch_first=True)
        # Fully Connected Layer 
        # 透過線性變換與 ReLU 激活函數，將 LSTM 的隱藏狀態映射回詞彙表維度
        self.linear = torch.nn.Sequential(torch.nn.Linear(in_features=hidden_dim,
                                                          out_features=hidden_dim),
                                          torch.nn.ReLU(),
                                          torch.nn.Linear(in_features=hidden_dim,
                                                          out_features=vocab_size))
        
    def forward(self, batch_x, batch_x_lens):
        """前向傳播"""
        return self.encoder(batch_x, batch_x_lens)
    
    def encoder(self, batch_x, batch_x_lens):
        """模型編碼邏輯，將字元序列轉化為預測機率"""
        # 向量化特徵提取
        batch_x = self.embedding(batch_x)
        # 針對變長序列進行優化，略過填充部分，提升計算效率
        batch_x = torch.nn.utils.rnn.pack_padded_sequence(batch_x,
                                                          batch_x_lens,
                                                          batch_first=True,
                                                          enforce_sorted=False)
        
        batch_x, _ = self.rnn_layer1(batch_x)
        batch_x, _ = self.rnn_layer2(batch_x)
        # D. 序列還原：將壓縮格式還原為標準張量，以便進入全連接層
        batch_x, _ = torch.nn.utils.rnn.pad_packed_sequence(batch_x,
                                                            batch_first=True)
        # 映射至字元機率分佈
        batch_x = self.linear(batch_x)
        return batch_x
    
    def generator(self, start_char, max_len=200):
           """
           給定起始字串，採自迴歸方式預測後續字元
           """
            # 將起始字元轉為 ID 列表
            char_list = [char_to_id[c] for c in start_char]
        
            next_char = None

            # 評估模式
            self.eval()

            # 使用torch.no_grad()上下文管理
            with torch.no_grad():
                while len(char_list) < max_len: 

                    # 1. 準備模型的輸入張量
                    device = next(self.parameters()).device
                    #    輸入需要有批次維度，所以我們將 char_list 包在另一個列表中
                    #    維度變為: [1, 當前序列長度]
                    input_tensor = torch.tensor([char_list], dtype=torch.long).to(device)
                
                    # 模型的 forward 方法也需要序列的實際長度
                    input_length = torch.tensor([len(char_list)], dtype=torch.long)
                
                    # 2. 將輸入傳入模型以獲得預測的logits
                    y = self.forward(input_tensor, input_length)
                
                    # 3. 我們只關心對「下一個」字元的預測，這對應於序列中「最後一個」時間點的輸出
                    last_time_step_logits = y[:, -1, :]
                
                    # 4. 使用argmax找出分數最高的字元的id
                    next_char = torch.argmax(last_time_step_logits, dim=1).item()
                

                    # 5. 檢查生成的字元是否為序列結束符號
                    if next_char == char_to_id['<eos>']:
                        break
                
                    # 6. 如果不是，將新字元的id加入到我們的列表中，並繼續迴圈
                    char_list.append(next_char)
            
            # 最終的 ID 列表轉換回字元，並回傳結果
            return [id_to_char[ch_id] for ch_id in char_list]

In [11]:
torch.manual_seed(2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = CharRNN(vocab_size,
                embed_dim,
                hidden_dim).to(device)

In [12]:
import torch.optim as optim
criterion = torch.nn.CrossEntropyLoss(ignore_index=char_to_id['<pad>'])
# 使用 Adam 優化器
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5) 

In [13]:
from tqdm import tqdm
from copy import deepcopy

# 設為訓練模式 
model = model.to(device)
model.train()

# i 控制何時印出 loss
i = 0 
for epoch in range(1, epochs+1):
    # 訓練迴圈
    bar = tqdm(dl_train, desc=f"Train epoch {epoch}")
    for batch_x, batch_y, batch_x_lens, batch_y_lens in bar:
        
        # 清除梯度:在計算新的梯度前，必須先清除上一步遺留的梯度
        optimizer.zero_grad()
    
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        # 模型前向傳播，得到預測結果
        # batch_pred_y 的維度: [batch_size, seq_len, vocab_size]
        batch_pred_y = model(batch_x, batch_x_lens)
        
        # 計算損失 & 反向傳播:CrossEntropyLoss要求 pred 維度為 (N, C) 和 target 維度為 (N)，所以要將batch維度和seq_len維度攤平
        pred_view = batch_pred_y.view(-1, vocab_size)
        target_view = batch_y.view(-1)
        
        loss = criterion(pred_view, target_view)
        
        # 損失計算梯度
        loss.backward()

        # gradient clipping防止梯度爆炸
        torch.nn.utils.clip_grad_value_(model.parameters(), grad_clip) 

        #更新模型參數：優化器根據計算出的梯度來更新模型的權重
        optimizer.step()

        i += 1
        if i % 50 == 0:
            bar.set_postfix(loss=loss.item())
    
    # 切到評估模式 
    model.eval()
    
    matched = 0
    total = 0
    bar_eval = tqdm(df_eval.iterrows(), desc=f"Validation epoch {epoch}")
    for _, row in bar_eval:
        # 從 DataFrame 中取出問題和標準答案
        batch_x = row['src']
        batch_y = str(row['tgt'])
        
        # 使用 generator 生成預測 
        prediction = model.generator(batch_x)
        
        # 答案是從 = 後面開始的
        try:
            equal_idx = prediction.index('=')
            predicted_answer = "".join(prediction[equal_idx+1:])
        except ValueError:
            predicted_answer = ""
            
        # 比較預測與標準答案
        if predicted_answer == batch_y:
            matched += 1
        
        total += 1
        
        # 更新進度條
        if total > 0:
            bar_eval.set_postfix(accuracy=f"{matched/total:.4f}")

    # 切回訓練，下一個 epoch
    model.train()
        
    print(f"\nEpoch {epoch} Validation Accuracy: {matched/total:.4f}")

Train epoch 1: 100%|██████████| 37020/37020 [04:52<00:00, 126.48it/s, loss=0.335]
Validation epoch 1: 263250it [39:48, 110.24it/s, accuracy=0.5793]



Epoch 1 Validation Accuracy: 0.5793


Train epoch 2: 100%|██████████| 37020/37020 [04:51<00:00, 126.80it/s, loss=0.228]
Validation epoch 2: 263250it [36:54, 118.87it/s, accuracy=0.7046]



Epoch 2 Validation Accuracy: 0.7046


Train epoch 3: 100%|██████████| 37020/37020 [04:52<00:00, 126.71it/s, loss=0.166]
Validation epoch 3: 263250it [37:10, 118.05it/s, accuracy=0.7002]


...


Epoch 8 Validation Accuracy: 0.8839


Train epoch 9: 100%|██████████| 37020/37020 [04:52<00:00, 126.60it/s, loss=0.103] 
Validation epoch 9: 263250it [37:21, 117.46it/s, accuracy=0.8662]



Epoch 9 Validation Accuracy: 0.8662


Train epoch 10: 100%|██████████| 37020/37020 [04:52<00:00, 126.76it/s, loss=0.0689]
Validation epoch 10: 263250it [37:36, 116.67it/s, accuracy=0.8927]



Epoch 10 Validation Accuracy: 0.8927
